<a href="https://colab.research.google.com/github/DaiyanNDahy/FlyRankInternML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaiyanNDahy/FlyRankInternML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents the daily performance metrics (e.g., impressions, clicks, average position) for a single content item (`content_hash_id`) within a specific client's context (`client_hash_id`).

**Table(s) used:** `FlyRank/internship-warehouse` dataset, specifically the `fact_content_daily_performance` table.

**Time Window:** A mid-panel month, such as March 2026, filtered by the `report_date` column.

**Predict/Rank (Label or Proxy):** We would predict or rank content based on future engagement, specifically `gsc_clicks` or `ga4_pageviews` (if GA4 data is available). A proxy for this could be a binary classification of whether content will achieve a 'high' Click-Through Rate (CTR) (e.g., above 0.15) within a future time window.

**One thing deliberately excluded:** I will deliberately exclude `client_hash_id` from direct use as a feature in a model that aims to generalize across clients, as its unique identifier nature would lead to overfitting or memorization of specific client performance rather than content characteristics.

In [24]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd
import numpy as np
import datetime # Added datetime import

# --- 1. Setup and Dataset Loading ---

# Load HF Token from Colab secrets. The prompt instructs to store it as 'HF_TOKEN'.
# This secret is essential for accessing the gated Hugging Face dataset.
HF_TOKEN = userdata.get('HF_TOKEN')

# Load the dataset in streaming mode as specified in the prompt.
# 'fact_content_daily_performance' is the specified dataset, split 'train'.
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

# --- 2. Filter for Mid-Panel Month (January 2025) and Initial Data Exploration ---

# Inspect the first few samples to determine actual column names, especially for date.
print("\n--- Inspecting Dataset Sample for Column Names ---")
sample_data = []
try:
    sample_data = list(ds.take(5))
except Exception as e:
    print(f"Error taking samples from dataset: {e}. Dataset might be empty or inaccessible.")

date_col_name = None
if sample_data:
    # Use keys from the first element of the streaming dataset directly
    first_element_keys = list(sample_data[0].keys())
    print(f"Keys found in first dataset element: {first_element_keys}")

    # Prioritize 'report_date' as it was found in the kernel state, then other common names.
    possible_date_cols = ['report_date', 'date', 'event_date', 'request_date', 'day', 'snapshot_date'] # Added 'report_date'
    for col in possible_date_cols:
        if col in first_element_keys:
            date_col_name = col
            break

    if date_col_name is None:
        print("Warning: No common date column name found (checked 'report_date', 'date', 'event_date', 'request_date', 'day', 'snapshot_date'). Please check dataset schema or manually specify the date column.")
        # Fallback to 'date' if no other candidate found, expecting an error if not present.
        date_col_name = 'date'
else:
    print("Warning: Dataset is empty or could not retrieve samples. Defaulting to 'date' for filtering, which might cause an error.")
    date_col_name = 'date' # Default to 'date' if sample is empty, might cause error if not present.

# Filter the streaming dataset for January 2025 data using the identified date column.
# Changed target month to January 2025 for faster execution based on sample data.
if date_col_name:
    print(f"Attempting to filter by column: '{date_col_name}' for January 2025")
    try:
        # Optimized filtering using datetime object attributes
        march_2026_ds_filtered = ds.filter(
            lambda x: x.get(date_col_name) and isinstance(x.get(date_col_name), datetime.date) and \
                      x.get(date_col_name).year == 2025 and x.get(date_col_name).month == 1
        )
    except Exception as e: # Catch a broader exception for safety
        print(f"Error during dataset filtering: {e}. It appears the column '{date_col_name}' is not accessible or has unexpected format.")
        print(f"Please inspect the actual structure of `ds` elements and the '{date_col_name}' values.")
        march_2026_ds_filtered = ds.filter(lambda x: False) # Return empty to prevent further errors
else:
    print("Cannot filter by date; date column name could not be determined. Returning empty filtered dataset.")
    march_2026_ds_filtered = ds.filter(lambda x: False) # Return empty if no date column

# Convert a sample of the filtered streaming dataset to a pandas DataFrame.
# Taking 1,000 rows as requested.
df_march_2026 = pd.DataFrame(list(march_2026_ds_filtered.take(1_000)))

# Ensure the 'date' column is in datetime format for proper analysis.
if date_col_name and date_col_name != 'date' and date_col_name in df_march_2026.columns:
    df_march_2026.rename(columns={date_col_name: 'date'}, inplace=True);
    print(f"Renamed '{date_col_name}' to 'date' for consistency.")
elif date_col_name == 'date' and 'date' not in df_march_2026.columns and not df_march_2026.empty:
    print(f"Warning: Expected 'date' column not found in df_march_2026 after taking samples. Data might be missing or date_col_name was inferred incorrectly.")

# Check if 'date' column exists after potential renaming or if it was initially named 'date'
if 'date' in df_march_2026.columns and not df_march_2026.empty:
    df_march_2026['date'] = pd.to_datetime(df_march_2026['date'])
    print(f"Converted 'date' column to datetime type.")
else:
    if not df_march_2026.empty:
        print("Error: 'date' column not found in DataFrame after loading and renaming. Cannot proceed with date-based analysis.")
    else:
        print("DataFrame is empty, skipping date column conversion.")

# --- 3. Verify Facts (Grain, Row Count, Date Span) ---

print("\n--- Verification Queries (January 2025) ---")

# Fact 1: Grain (Unit of analysis)
# The unit of analysis is expected to be daily performance for a single content item.
# Assuming 'content_hash_id' is the unique identifier for content items within the dataset.
# Check if (date, content_hash_id) forms a unique key for each row.
content_id_col = 'content_hash_id' # Based on observed keys

if content_id_col in df_march_2026.columns and 'date' in df_march_2026.columns and not df_march_2026.empty:
    grain_cols = ['date', content_id_col]
    num_rows = len(df_march_2026)
    num_unique_grains = df_march_2026[grain_cols].drop_duplicates().shape[0]

    print(f"\nFact 1: Grain Verification")
    print(f"      Total rows in January 2025 sample: {num_rows}")
    print(f"      Unique (date, {content_id_col}) combinations: {num_unique_grains}")

    if num_unique_grains == num_rows:
        print(f"      Conclusion: One row indeed represents the daily performance of a single content item (date, {content_id_col}).")
    else:
        print(f"      Conclusion: Discrepancy found. The grain might be finer, or there are duplicate entries for (date, {content_id_col}).")
elif not df_march_2026.empty:
    print(f"Warning: '{content_id_col}' or 'date' column not found, or DataFrame is empty. Cannot verify grain.")
    print("Available columns: ", df_march_2026.columns.tolist())
else:
    print("DataFrame is empty, skipping grain verification.")

# Fact 2: Slice's row count and date span
print(f"\nFact 2: Row Count and Date Span")
if not df_march_2026.empty and 'date' in df_march_2026.columns:
    print(f"      Total rows in the sampled data (January 2025): {len(df_march_2026)}")
    print(f"      Date span: {df_march_2026['date'].min().strftime('%Y-%m-%d')} to {df_march_2026['date'].max().strftime('%Y-%m-%d')}")
else:
    print("      DataFrame is empty or 'date' column is missing, cannot provide row count and date span.")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]


--- Inspecting Dataset Sample for Column Names ---
Keys found in first dataset element: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Attempting to filter by column: 'report_date' for January 2025
Renamed 'report_date' to 'date' for consistency.
Converted 'date' column to datetime type.

--- Verification Queries (January 2025) ---

Fact 1: Grain Verification
      Total rows in January 2025 sample: 1000
      Unique (date, content_hash_id) combinations: 1000
      Conclusion: One row indeed represents the daily pe

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature Fields:**
*   `day_of_week` (derived from `report_date`)
*   `is_weekend` (derived from `report_date`)
*   `gsc_ctr` (derived from `gsc_clicks` / `gsc_impressions`)
*   `gsc_log_impressions` (derived from `gsc_impressions`)
*   `gsc_avg_position`

**Label / Proxy Field:**
*   `is_high_performing_content_next_day` (Hypothetical binary label: e.g., will next day's CTR be > 0.15?)

**Context Fields:**
*   `report_date`
*   `client_hash_id`
*   `content_hash_id`
*   `client_has_gsc`
*   `client_has_ga4`
*   `gsc_data_available`
*   `ga4_data_available`

**Excluded Fields (with why):**
*   `gsc_sum_position`: This is a sum and `gsc_avg_position` is usually more informative for ranking models. Redundant.
*   `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`: These are GA4 metrics. While valuable, for a model primarily focused on Google Search Console (GSC) performance or to simplify the initial scope, these can be excluded to avoid dealing with differing data availability and definitions.
*   `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`, `scroll_events`: These columns provide more granular detail on traffic sources and AI bot activity. For an initial content ranking model, they might add complexity without a clear immediate benefit to the core task of predicting content performance from standard GSC metrics. Could be revisited for richer models.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Ensure df_march_2026 is available from the previous cell (LeOVCmbxyFKq)
if 'df_march_2026' not in locals() or df_march_2026.empty:
    print("Error: df_march_2026 not found or is empty. Please run the previous cell first and ensure data is loaded.")
    # Attempt to create a minimal df for demonstration if the previous cell wasn't run/failed
    df_march_2026 = pd.DataFrame({
        'date': pd.to_datetime([]),
        'content_hash_id': [],
        'gsc_data_available': []
    })

print("\n--- Verification Queries (January 2025) ---")

# Fact 3: Availability (filter with IS TRUE)
# Demonstrating filtering based on a boolean column, e.g., 'gsc_data_available'.
# This shows how many rows survive under a specific condition.
if 'gsc_data_available' in df_march_2026.columns:
    initial_rows = len(df_march_2026)
    filtered_df = df_march_2026[df_march_2026['gsc_data_available'] == True] # Using '== True' for explicit IS TRUE check
    surviving_rows = len(filtered_df)
    print(f"\nFact 3: Availability Check (gsc_data_available IS TRUE)")
    print(f"      Initial rows (January 2025 sample): {initial_rows}")
    print(f"      Rows where 'gsc_data_available' is TRUE: {surviving_rows}")
    print(f"      Percentage of rows with GSC data available: {surviving_rows / initial_rows * 100:.2f}%" if initial_rows > 0 else "      No initial rows.")
elif 'gsc_clicks' in df_march_2026.columns:
    # Fallback if 'gsc_data_available' is not present but GSC clicks are available
    initial_rows = len(df_march_2026)
    filtered_df = df_march_2026[df_march_2026['gsc_clicks'] > 0]
    surviving_rows = len(filtered_df)
    print(f"\nFact 3: Fallback Availability Check (gsc_clicks > 0 IS TRUE)")
    print(f"      Initial rows (January 2025 sample): {initial_rows}")
    print(f"      Rows where 'gsc_clicks' > 0: {surviving_rows}")
    print(f"      Percentage of rows with positive GSC clicks: {surviving_rows / initial_rows * 100:.2f}%" if initial_rows > 0 else "      No initial rows.")
else:
    print("Warning: 'gsc_data_available' or 'gsc_clicks' column not found for availability check.")
    print("      No suitable boolean-like column or GSC metrics found for availability check.")



--- Verification Queries (January 2025) ---

Fact 3: Availability Check (gsc_data_available IS TRUE)
      Initial rows (January 2025 sample): 1000
      Rows where 'gsc_data_available' is TRUE: 1000
      Percentage of rows with GSC data available: 100.00%


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd # Ensure pandas is imported
import numpy as np # Ensure numpy is imported

# Ensure df_march_2026 is available from the previous cell (LeOVCmbxyFKq)
if 'df_march_2026' not in locals() or df_march_2026.empty:
    print("Error: df_march_2026 not found or is empty. Please run the previous cell first and ensure data is loaded.")
    # Attempt to create a minimal df for demonstration if the previous cell wasn't run/failed
    df_march_2026 = pd.DataFrame({
        'date': pd.to_datetime(['2025-01-01']),
        'content_hash_id': ['test_content_id'],
        'gsc_impressions': [100],
        'gsc_clicks': [10],
        'gsc_avg_position': [5.0]
    })

print("\n--- Five Features ---")
# Build a small feature frame (can be derived columns on df_march_2026)
# Every feature needs to be knowable at the decision moment.

# Create a copy to avoid SettingWithCopyWarning when adding new columns
feature_df = df_march_2026.copy()

# Check if feature_df is empty before proceeding with feature engineering
if feature_df.empty:
    print("DataFrame is empty, skipping feature engineering.")
else:
    # Feature 1: Day of the week (0=Monday, 6=Sunday)
    feature_df['day_of_week'] = feature_df['date'].dt.dayofweek
    print("      Feature 1: 'day_of_week' - Knowable at the decision moment because it's derived from the date.")

    # Feature 2: Is Weekend
    feature_df['is_weekend'] = feature_df['day_of_week'].isin([5, 6]).astype(int)
    print("      Feature 2: 'is_weekend' - Knowable at the decision moment because it's derived from the date.")

    # Feature 3: Click-Through Rate (CTR) - using gsc_clicks and gsc_impressions
    if 'gsc_clicks' in feature_df.columns and 'gsc_impressions' in feature_df.columns:
        feature_df['gsc_ctr'] = feature_df['gsc_clicks'] / feature_df['gsc_impressions']
        # Handle potential division by zero by filling NaN/Inf with 0
        feature_df['gsc_ctr'] = feature_df['gsc_ctr'].replace([np.inf, -np.inf], np.nan).fillna(0)
        print("      Feature 3: 'gsc_ctr' (gsc_clicks / gsc_impressions) - Knowable at the decision moment because it's a ratio of current day's observed metrics.")
    else:
        print("      Warning: 'gsc_clicks' or 'gsc_impressions' column not found for 'gsc_ctr' feature. Setting to 0.")
        feature_df['gsc_ctr'] = 0.0 # Placeholder

    # Feature 4: Logged Impressions (to handle skewed distribution) - using gsc_impressions
    if 'gsc_impressions' in feature_df.columns:
        # Add a small constant (1) to handle zero impressions for log transform
        feature_df['gsc_log_impressions'] = np.log1p(feature_df['gsc_impressions'])
        print("      Feature 4: 'gsc_log_impressions' - Knowable at the decision moment because it's a transformation of current day's observed impressions.")
    else:
        print("      Warning: 'gsc_impressions' column not found for 'gsc_log_impressions' feature. Setting to 0.")
        feature_df['gsc_log_impressions'] = 0.0 # Placeholder

    # Feature 5: Average position - using gsc_avg_position
    if 'gsc_avg_position' in feature_df.columns:
        feature_df['gsc_avg_position_rank'] = feature_df['gsc_avg_position'] # Use directly as it's already an average/single value
        print("      Feature 5: 'gsc_avg_position_rank' - Knowable at the decision moment because it reflects the content's visibility from current day's observation.")
    else:
        print("      Warning: 'gsc_avg_position' column not found for 'gsc_avg_position_rank' feature. Setting to 0.")
        # Fallback to a placeholder or another simple derivable feature if 'gsc_avg_position' is missing.
        feature_df['day_of_month'] = feature_df['date'].dt.day # Fallback feature
        print("      Fallback Feature 5: 'day_of_month' - Knowable at the decision moment because it's derived from the date.")

    print("\n--- First few rows of the feature frame ---")
    # Display relevant feature columns along with identifiers
    display_cols = ['date', 'content_hash_id', 'day_of_week', 'is_weekend', 'gsc_ctr', 'gsc_log_impressions']
    if 'gsc_avg_position_rank' in feature_df.columns:
        display_cols.append('gsc_avg_position_rank')
    elif 'day_of_month' in feature_df.columns:
        display_cols.append('day_of_month')

    display(feature_df[display_cols].head())



--- Five Features ---
      Feature 1: 'day_of_week' - Knowable at the decision moment because it's derived from the date.
      Feature 2: 'is_weekend' - Knowable at the decision moment because it's derived from the date.
      Feature 3: 'gsc_ctr' (gsc_clicks / gsc_impressions) - Knowable at the decision moment because it's a ratio of current day's observed metrics.
      Feature 4: 'gsc_log_impressions' - Knowable at the decision moment because it's a transformation of current day's observed impressions.
      Feature 5: 'gsc_avg_position_rank' - Knowable at the decision moment because it reflects the content's visibility from current day's observation.

--- First few rows of the feature frame ---


,date,content_hash_id,day_of_week,is_weekend,gsc_ctr,gsc_log_impressions,gsc_avg_position_rank
0,2025-01-27,content_3b70a18ea133b2bb,0,0,0.0,3.433987,3.833333
1,2025-01-27,content_fe8e8155ce1d47a2,0,0,0.0,1.791759,71.600000
2,2025-01-27,content_b4462a1b90640058,0,0,0.0,0.693147,34.000000
3,2025-01-27,content_c899aef92518c714,0,0,0.0,1.945910,23.333333
4,2025-01-27,content_c7c1d2e68d9d0964,0,0,0.0,1.791759,17.800000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One Named Limitation:** The dataset primarily focuses on Search Console and Analytics data related to content performance. It lacks direct information about off-platform factors, such as social media trends, competitor activities, or external news events, which could significantly influence content performance but are not captured in this slice. Therefore, a model built on this data alone might not fully capture external drivers of content success.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Ensure feature_df is available from the previous cell (wdk3tsQiyFKt)
if 'feature_df' not in locals() or feature_df.empty:
    print("Error: feature_df not found or is empty. Please run the previous cell first and ensure data is loaded.")
    # Attempt to create a minimal df for demonstration if the previous cell wasn't run/failed
    feature_df = pd.DataFrame({
        'date': pd.to_datetime(['2026-03-01', '2026-03-01', '2026-03-02']),
        'content_hash_id': ['a', 'b', 'a'],
        'gsc_clicks': [10, 50, 15],
        'gsc_impressions': [100, 200, 150]
    })
    feature_df['gsc_ctr'] = feature_df['gsc_clicks'] / feature_df['gsc_impressions']

print("\n--- The Trap: Deliberate Leakage Experiment ---")

# The goal is to demonstrate data leakage by creating a label-derived column
# that provides information about the target variable that would not be available
# at the time of prediction.

# Step 1: Define a hypothetical 'label'. For example, 'is_high_performing_content'.
# This label could be defined as content having a CTR above a certain threshold.
# Assuming 'gsc_ctr' is calculated in the previous step.
if 'gsc_ctr' in feature_df.columns:
    CTR_THRESHOLD = 0.15 # Example threshold for high performance
    feature_df['is_high_performing_label'] = (feature_df['gsc_ctr'] > CTR_THRESHOLD).astype(int)
    print(f"1. Hypothetical 'label' created: 'is_high_performing_label' (GSC CTR > {CTR_THRESHOLD}).")
    print("   First few values of the label:")
    display(feature_df[['gsc_ctr', 'is_high_performing_label']].head())
else:
    print("Warning: 'gsc_ctr' column not found. Cannot create 'is_high_performing_label'. Skipping leakage demo.")


# Step 2: Create a 'leaking feature' that is directly derived from the label.
# This feature would give away information about the label *before* the decision moment.
if 'is_high_performing_label' in feature_df.columns:
    feature_df['leaky_future_performance_signal'] = feature_df['is_high_performing_label']
    print("\n2. Deliberate 'leaking feature' created: 'leaky_future_performance_signal'.")
    print("   This feature directly duplicates information from the label, causing leakage.")
    print("   If a model were trained with this feature, its performance (e.g., accuracy) would be artificially high, but not generalize to future data.")
    print("   First few values of the leaking feature:")
    display(feature_df[['is_high_performing_label', 'leaky_future_performance_signal']].head())

    # Step 3: Remove the leaking feature, as it would not be available in a real-world scenario.
    feature_df = feature_df.drop(columns=['leaky_future_performance_signal'])
    print("\n3. Leaking feature 'leaky_future_performance_signal' has been removed.")
    print("   This demonstrates the importance of identifying and removing features that leak future information, as they lead to overly optimistic performance estimates.")
else:
    print("Skipping leakage feature creation and removal as 'is_high_performing_label' was not created.")

# --- 5. One Named Limitation of Your Slice (This is also covered in markdown cell 4) ---

print("\n--- One Named Limitation (from markdown cell above) ---")
print("Limitation: The dataset primarily focuses on Search Console and Analytics data related to content performance.")
print("It lacks direct information about off-platform factors, such as social media trends, competitor activities, or external news events, which could significantly influence content performance but are not captured in this slice.")



--- The Trap: Deliberate Leakage Experiment ---
1. Hypothetical 'label' created: 'is_high_performing_label' (GSC CTR > 0.15).
   First few values of the label:


,gsc_ctr,is_high_performing_label
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0



2. Deliberate 'leaking feature' created: 'leaky_future_performance_signal'.
   This feature directly duplicates information from the label, causing leakage.
   If a model were trained with this feature, its performance (e.g., accuracy) would be artificially high, but not generalize to future data.
   First few values of the leaking feature:


,is_high_performing_label,leaky_future_performance_signal
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0



3. Leaking feature 'leaky_future_performance_signal' has been removed.
   This demonstrates the importance of identifying and removing features that leak future information, as they lead to overly optimistic performance estimates.

--- One Named Limitation (from markdown cell above) ---
Limitation: The dataset primarily focuses on Search Console and Analytics data related to content performance.
It lacks direct information about off-platform factors, such as social media trends, competitor activities, or external news events, which could significantly influence content performance but are not captured in this slice.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.